In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import warnings

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
sns.set_style("whitegrid")

# Define file patterns
# specific file name for unemployment
unemp_filename_search = 'US unemployment rate*' 
# broader pattern for compensation files (2019-2024)
comp_file_pattern = 'national_M20*_dl*' 

# ==========================================
# 2. LOAD & PROCESS UNEMPLOYMENT DATA
# ==========================================
print("--- Loading Unemployment Data ---")

# Find the file
unemp_files = glob.glob(unemp_filename_search)
if not unemp_files:
    print("Error: Could not find 'US unemployment rate' file.")
    df_unemp_long = pd.DataFrame()
else:
    unemp_file = unemp_files[0]
    print(f"Processing: {unemp_file}")

    try:
        # Try reading as Excel first (as requested)
        # Skip 11 rows of metadata usually found in BLS files
        try:
            df_unemp = pd.read_excel(unemp_file, skiprows=11)
        except Exception:
            # Fallback to CSV if Excel fails
            df_unemp = pd.read_csv(unemp_file, skiprows=11)

        # Clean Data
        # Ensure Year is numeric and filter for 2019-2024
        if 'Year' in df_unemp.columns:
            df_unemp['Year'] = pd.to_numeric(df_unemp['Year'], errors='coerce')
            df_unemp = df_unemp.dropna(subset=['Year'])
            df_unemp['Year'] = df_unemp['Year'].astype(int)
            df_unemp = df_unemp[df_unemp['Year'].between(2019, 2024)].copy()

            # Reshape from Wide (Jan, Feb...) to Long (Date, Rate) format
            df_unemp_long = df_unemp.melt(
                id_vars=['Year'], 
                value_vars=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
                var_name='Month_Name', 
                value_name='Unemployment Rate'
            )

            # Convert Month Names to Numbers
            month_map = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
                         'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
            df_unemp_long['Month'] = df_unemp_long['Month_Name'].map(month_map)
            df_unemp_long['Day'] = 1

            # Create proper Datetime column
            df_unemp_long['Date'] = pd.to_datetime(df_unemp_long[['Year', 'Month', 'Day']].rename(columns={'Year':'year', 'Month':'month', 'Day':'day'}))
            df_unemp_long = df_unemp_long.sort_values('Date').reset_index(drop=True)
            print("  Successfully loaded unemployment data.")
        else:
            print("  Error: 'Year' column not found in unemployment file.")
            df_unemp_long = pd.DataFrame()

    except Exception as e:
        print(f"  Error reading unemployment file: {e}")
        df_unemp_long = pd.DataFrame()

# ==========================================
# 3. LOAD & PROCESS COMPENSATION DATA
# ==========================================
print("\n--- Loading Compensation Data ---")
comp_data = []
found_files = glob.glob(comp_file_pattern)

# Sort files to process in order
found_files = sorted(found_files)

for f in found_files:
    # Skip metadata/description files
    if 'Field' in f or 'Update' in f or 'Descriptions' in f:
        continue
        
    try:
        # Extract year from filename (looks for 'M2019', 'M2020' etc)
        year = None
        for y in range(2019, 2025):
            if str(y) in f:
                year = y
                break
        
        if year:
            try:
                try:
                    df_comp = pd.read_excel(f)
                except:
                    df_comp = pd.read_csv(f, low_memory=False)
            except UnicodeDecodeError:
                df_comp = pd.read_csv(f, encoding='latin1', low_memory=False)
            
            # Normalize columns to lowercase (e.g., OCC_CODE -> occ_code)
            df_comp.columns = df_comp.columns.str.lower().str.strip()
            
            # Filter for "All Occupations" (Standard code: 00-0000)
            if 'occ_code' in df_comp.columns:
                row = df_comp[df_comp['occ_code'] == '00-0000']
                
                if not row.empty:
                    h_mean = row['h_mean'].values[0]
                    
                    # Clean string values (e.g. remove '$' or ',')
                    if isinstance(h_mean, str):
                        h_mean = float(h_mean.replace(',', '').replace('*', '').replace('#', ''))
                    
                    if pd.notnull(h_mean):
                        comp_data.append({'Year': year, 'Hourly_Mean_Wage': h_mean})
                        print(f"  Processed {year}: ${h_mean}/hr (from {os.path.basename(f)})")
            else:
                print(f"  Skipping {f}: 'occ_code' column not found.")
                
    except Exception as e:
        print(f"  Error processing {f}: {e}")

# ==========================================
# 4. MERGE & VISUALIZE
# ==========================================
if not comp_data or df_unemp_long.empty:
    print("\nError: insufficient data to generate plots.")
else:
    # Convert comp data to DataFrame
    df_comp_annual = pd.DataFrame(comp_data).sort_values('Year')
    
    # Create a monthly timeline to interpolate annual wage data
    # (Using May 1st as the standard reference date for OEWS)
    df_comp_annual['Date'] = pd.to_datetime(df_comp_annual['Year'].astype(str) + '-05-01')

    # Create skeleton DataFrame for all months
    all_months = pd.DataFrame({'Date': pd.date_range(start='2019-01-01', end='2024-12-01', freq='MS')})

    # Merge and Interpolate
    df_wages_interp = all_months.merge(df_comp_annual[['Date', 'Hourly_Mean_Wage']], on='Date', how='left')
    df_wages_interp = df_wages_interp.set_index('Date')
    df_wages_interp['Hourly_Mean_Wage'] = df_wages_interp['Hourly_Mean_Wage'].interpolate(method='time')
    # Fill remaining NaNs (start/end) using forward/backward fill
    df_wages_interp['Hourly_Mean_Wage'] = df_wages_interp['Hourly_Mean_Wage'].ffill().bfill()
    df_wages_interp = df_wages_interp.reset_index()

    # Final Merge with Unemployment
    df_final = pd.merge(df_unemp_long, df_wages_interp, on='Date', how='inner')

    # Calculate Wage Growth (Year-over-Year)
    df_final['Wage_Growth_YoY'] = df_final['Hourly_Mean_Wage'].pct_change(12) * 100

    # --- Chart 1: Unemployment vs. Compensation Trends ---
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Left Axis: Unemployment
    color_unemp = '#d62728' # Red
    ax1.set_xlabel('Date', fontsize=12)
    ax1.set_ylabel('Unemployment Rate (%)', color=color_unemp, fontsize=12, fontweight='bold')
    ax1.plot(df_final['Date'], df_final['Unemployment Rate'], color=color_unemp, linewidth=3, label='Unemployment Rate')
    ax1.tick_params(axis='y', labelcolor=color_unemp)
    ax1.set_ylim(0, 16) # Scale for 2020 spike

    # Right Axis: Compensation
    ax2 = ax1.twinx()
    color_wage = '#1f77b4' # Blue
    ax2.set_ylabel('Avg. Hourly Wage ($)', color=color_wage, fontsize=12, fontweight='bold')
    ax2.plot(df_final['Date'], df_final['Hourly_Mean_Wage'], color=color_wage, linewidth=3, linestyle='--', label='Avg Hourly Wage')
    ax2.tick_params(axis='y', labelcolor=color_wage)

    # Plot actual annual data points
    ax2.scatter(df_comp_annual['Date'], df_comp_annual['Hourly_Mean_Wage'], color='navy', s=80, zorder=5, label='Annual OEWS Data')

    plt.title('US Unemployment Trends vs. Average Compensation (2019-2024)', fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()

    # --- Chart 2: Phillips Curve (Wage Growth vs. Unemployment) ---
    plt.figure(figsize=(10, 6))
    df_scatter = df_final.dropna(subset=['Wage_Growth_YoY'])

    sns.scatterplot(
        data=df_scatter, 
        x='Unemployment Rate', 
        y='Wage_Growth_YoY', 
        hue=df_scatter['Date'].dt.year, 
        palette='viridis', 
        s=100,
        edgecolor='black'
    )

    plt.title('Wage Growth vs. Unemployment (The Phillips Curve)', fontsize=14)
    plt.xlabel('Unemployment Rate (%)', fontsize=12)
    plt.ylabel('Annual Wage Growth (%)', fontsize=12)
    plt.legend(title='Year')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()